# CE49X Midterm Exam - Part 2 SOLUTION KEY
## Power Grid Stability Prediction

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Date:** April 8, 2026

---

**This notebook contains complete solutions and rubric annotations.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

---
## Task 1: Data Loading & Exploration (8 pts)

In [ ]:
# Load the dataset
df = pd.read_csv('data/electrical_grid_stability.csv')

# Shape and dtypes
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

In [ ]:
# First 5 rows
print("First 5 rows:")
df.head()

In [ ]:
# Missing values
print(f"Missing values per column:\n{df.isnull().sum()}")
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Describe
df.describe()

In [ ]:
# Value counts of stabf
print("Target variable distribution:")
print(df['stabf'].value_counts())
print(f"\nPercentages:\n{df['stabf'].value_counts(normalize=True) * 100}")

### RUBRIC - Task 1
- **2 pts:** Data loaded correctly, shape (10000, 14) and dtypes printed
- **1 pt:** `df.head()` displayed
- **2 pts:** Missing values checked (should be 0). Accept `df.isnull().sum()`, `df.isna().sum()`, or `df.info()`
- **1 pt:** `.describe()` printed
- **2 pts:** `stabf` value counts printed. Should show ~3620 stable, ~6380 unstable (approximately 36%/64%)

---
## Task 2: Feature Engineering (8 pts)

In [ ]:
# Convert stabf to numeric
df['is_unstable'] = (df['stabf'] == 'unstable').astype(int)

# Create new features
df['total_reaction_time'] = df['tau1'] + df['tau2'] + df['tau3'] + df['tau4']
df['power_imbalance'] = df['p1'] + df['p2'] + df['p3'] + df['p4']
df['avg_elasticity'] = df[['g1', 'g2', 'g3', 'g4']].mean(axis=1)

print("New columns added:")
print(df[['is_unstable', 'total_reaction_time', 'power_imbalance', 'avg_elasticity']].head())

In [ ]:
# Class balance
counts = df['is_unstable'].value_counts()
pcts = df['is_unstable'].value_counts(normalize=True) * 100
print("Class Balance:")
print(f"  Stable (0):   {counts[0]:,} ({pcts[0]:.1f}%)")
print(f"  Unstable (1): {counts[1]:,} ({pcts[1]:.1f}%)")

### RUBRIC - Task 2
- **2 pts:** `is_unstable` created correctly (1 for unstable, 0 for stable). Accept `df['stabf'].map({'stable': 0, 'unstable': 1})` or equivalent
- **2 pts:** `total_reaction_time` = sum of tau1-tau4
- **1 pt:** `power_imbalance` = sum of p1-p4
- **1 pt:** `avg_elasticity` = mean of g1-g4
- **2 pts:** Class balance printed with both counts and percentages

---
## Task 3: Grouped Analysis (10 pts)

In [ ]:
# Grouped means by stabf
original_features = ['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4',
                     'g1', 'g2', 'g3', 'g4']

grouped = df.groupby('stabf')[original_features].mean()
print("Mean of features grouped by stability:")
print(grouped.T)  # Transpose for readability

# Which differ most?
diff = (grouped.loc['unstable'] - grouped.loc['stable']).abs().sort_values(ascending=False)
print(f"\nFeatures with largest absolute difference between groups:")
print(diff.head(5))

In [ ]:
# Correlation with stab (continuous target)
corr_with_stab = df[original_features].corrwith(df['stab']).abs().sort_values(ascending=False)
print("Absolute correlation of features with stab:")
print(corr_with_stab)
print(f"\nTop 3 most correlated features: {corr_with_stab.head(3).index.tolist()}")

In [ ]:
# Unstable grids only: tau1 statistics
unstable = df[df['stabf'] == 'unstable']
print(f"Unstable grids - tau1 statistics:")
print(f"  Min:  {unstable['tau1'].min():.4f}")
print(f"  Max:  {unstable['tau1'].max():.4f}")
print(f"  Mean: {unstable['tau1'].mean():.4f}")

In [ ]:
# Compare g1 between stable and unstable
g1_stats = df.groupby('stabf')['g1'].agg(['mean', 'std'])
print("g1 (producer elasticity) by stability:")
print(g1_stats)

### RUBRIC - Task 3
- **3 pts:** Grouped means computed correctly. Must mention which features differ most (1 pt for computation, 1 pt for identifying top differing features, 1 pt for brief discussion)
- **3 pts:** Correlation with `stab` computed. Top 3 identified correctly. Accept `.corr()` matrix approach or `.corrwith()`. Must use absolute values for ranking
- **2 pts:** Unstable subset filtered correctly, tau1 min/max/mean reported
- **2 pts:** g1 compared between groups with mean and std

---
## Task 4: Visualization (12 pts)

In [ ]:
# (a) Boxplot of tau1 by stabf
fig, ax = plt.subplots(figsize=(8, 5))

stable_tau1 = df[df['stabf'] == 'stable']['tau1']
unstable_tau1 = df[df['stabf'] == 'unstable']['tau1']

bp = ax.boxplot([stable_tau1, unstable_tau1], labels=['Stable', 'Unstable'],
                patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('indianred')

ax.set_xlabel('Grid Stability', fontsize=12)
ax.set_ylabel('Producer Reaction Time (seconds)', fontsize=12)
ax.set_title('Unstable Grids Have Higher Producer Reaction Times', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# (b) Scatter plot of tau1 vs g1, colored by stabf
fig, ax = plt.subplots(figsize=(8, 6))

stable = df[df['stabf'] == 'stable']
unstable = df[df['stabf'] == 'unstable']

ax.scatter(stable['tau1'], stable['g1'], alpha=0.3, s=10,
           color='steelblue', label='Stable')
ax.scatter(unstable['tau1'], unstable['g1'], alpha=0.3, s=10,
           color='indianred', label='Unstable')

ax.set_xlabel('Producer Reaction Time, tau1 (seconds)', fontsize=12)
ax.set_ylabel('Producer Price Elasticity, g1', fontsize=12)
ax.set_title('Producer Reaction Time vs Price Elasticity by Grid Stability', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# (c) Correlation heatmap of 12 original features
corr_matrix = df[original_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)

# Labels
ax.set_xticks(range(len(original_features)))
ax.set_yticks(range(len(original_features)))
ax.set_xticklabels(original_features, rotation=45, ha='right')
ax.set_yticklabels(original_features)

# Annotations
for i in range(len(original_features)):
    for j in range(len(original_features)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=7,
                color='white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black')

ax.set_title('Correlation Matrix of Grid Features', fontsize=13)
plt.colorbar(im, label='Correlation')
plt.tight_layout()
plt.show()

### RUBRIC - Task 4

**(a) Boxplot (4 pts):**
- 2 pts: Correct boxplot with tau1 grouped by stabf
- 1 pt: Axis labels present
- 1 pt: Title states a finding (e.g., "Unstable grids have higher reaction times"). Deduct if title is just "Boxplot of tau1"

**(b) Scatter (4 pts):**
- 2 pts: Correct scatter plot of tau1 vs g1
- 1 pt: Points colored by stability class
- 1 pt: Legend and axis labels present

**(c) Heatmap (4 pts):**
- 2 pts: Correct correlation matrix of the 12 features
- 1 pt: Sequential or diverging colormap used (not jet/rainbow)
- 1 pt: Feature labels readable. Annotations optional but award if present

**Common deductions:** -1 per plot if missing axis labels or units

---
## Task 5: Statistical Analysis (6 pts)

In [ ]:
# Z-scores for tau1
mean_tau1 = df['tau1'].mean()
std_tau1 = df['tau1'].std()
df['tau1_zscore'] = (df['tau1'] - mean_tau1) / std_tau1

outliers = df[df['tau1_zscore'].abs() > 2]
print(f"tau1 mean: {mean_tau1:.4f}, std: {std_tau1:.4f}")
print(f"Number of samples with |z| > 2: {len(outliers)}")
print(f"Percentage: {len(outliers)/len(df)*100:.1f}%")

In [ ]:
# Two-sample t-test: tau1 stable vs unstable
stable_tau1 = df[df['stabf'] == 'stable']['tau1']
unstable_tau1 = df[df['stabf'] == 'unstable']['tau1']

t_stat, p_value = stats.ttest_ind(stable_tau1, unstable_tau1)
print(f"Two-sample t-test for tau1:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.6f}")
print(f"  Significant at alpha=0.05? {'Yes' if p_value < 0.05 else 'No'}")

### Interpretation

**H0:** The mean tau1 is the same for stable and unstable grids (no difference in producer reaction time between groups).  
**H1:** The mean tau1 differs between stable and unstable grids.

**Result:** The p-value is extremely small (< 0.05), so we reject H0. There is statistically significant evidence that producer reaction time differs between stable and unstable grid configurations.

**Most predictive feature:** Based on correlation analysis, the features with the highest absolute correlation with the continuous stability measure (`stab`) are the elasticity coefficients (`g1`-`g4`) and reaction times (`tau1`-`tau4`). The specific top feature will depend on the correlation values computed above.

### RUBRIC - Task 5
- **2 pts:** Z-scores computed correctly. Number of outliers (|z| > 2) reported. Accept `scipy.stats.zscore()` or manual calculation
- **2 pts:** T-test performed correctly. H0/H1 stated. p-value reported. Correct interpretation at alpha = 0.05
- **2 pts:** Most predictive feature identified with numerical justification (correlation value, group mean difference, or similar). Accept any reasonable answer if supported by data

---
## Task 6: Classification (6 pts)

In [ ]:
# Define X and y
X = df[original_features]
y = df['is_unstable']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Scale — fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)

# Metrics for the unstable class (pos_label=1)
print("Classification Results (Unstable class):")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred):.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"                Pred Stable  Pred Unstable")
print(f"  Act Stable      {cm[0,0]:>5}        {cm[0,1]:>5}")
print(f"  Act Unstable    {cm[1,0]:>5}        {cm[1,1]:>5}")

### RUBRIC - Task 6
- **1 pt:** Correct train-test split with `stratify=y`, `test_size=0.2`, `random_state=42`
- **1 pt:** StandardScaler fit on train only, transform on both. Deduct if `fit_transform` on full dataset
- **1 pt:** Model trained and predictions made
- **2 pts:** All four metrics reported (accuracy, precision, recall, F1). Deduct 0.5 per missing metric
- **1 pt:** Confusion matrix printed in readable format

---
## Written Questions — Solutions

### Written Question 1 (3 pts) — Solution

**False Stable is more dangerous.** If the model predicts a grid configuration is stable when it is actually unstable, operators will not take corrective action, potentially leading to blackouts, equipment damage, and cascading infrastructure failures. A False Unstable (false alarm) only leads to unnecessary precautionary measures — costly but not dangerous.

**Recall** for the "unstable" class should be prioritized, because we want to catch as many truly unstable configurations as possible (minimize false negatives), even at the cost of some false alarms.

**RUBRIC:**
- 1 pt: Correctly identifies False Stable as more dangerous
- 1 pt: Reasonable engineering justification (blackouts, safety, cascading failures)
- 1 pt: States recall should be prioritized with correct reasoning

### Written Question 2 — BONUS (3 pts) — Solution

**No, correlation does not prove causation.** Correlation tells us that two variables tend to move together, but it does not tell us which one causes the other, or whether a third variable causes both.

**Example confounding variable:** Grid age or infrastructure quality. Older grids may have both slower-reacting producers (aging equipment) and more instability (degraded infrastructure). In that case, age would be the true cause of both, creating a spurious correlation between reaction time and instability.

Other acceptable confounders: grid load level, weather conditions, equipment maintenance schedules.

**RUBRIC:**
- 1 pt: States correlation does not equal causation
- 1 pt: Explains the distinction clearly
- 1 pt: Names a plausible confounding variable with explanation

---

### End of Solution Key

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr